# Mock-pipeline validation

A minimal end-to-end BayMorph pipeline example on a controlled mock observation.

This notebook generates a noisy single-Sérsic mock, writes it to FITS, runs image preparation, runs the real-fit pipeline, and compares the recovered parameters with the truth.


## Workflow

```text
Truth parameters
        ↓
Generate mock observation
        ↓
Run image preparation
        ↓
Bayesian fit
        ↓
Posterior analysis
        ↓
Corner plot
        ↓
Mock observation report
        ↓
Fit diagnostics
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

from baymorph.simulation.sersic import SyntheticSersic
from baymorph.simulation.observation import make_observation
from baymorph.pipeline.workflows import run_image_preparation, run_real_fit
from baymorph.pipeline.image_preparation import load_image_preparation_products
from baymorph.plotting import plot_fit_summary

In [ ]:
shape = (101, 101)
output_root = Path("notebooks/validation_outputs/mock_pipeline")
image_dir = output_root / "input_images"
image_dir.mkdir(parents=True, exist_ok=True)

truth = {
    "flux": 1.0e5,
    "re": 8.0,
    "n": 2.0,
    "q": 0.65,
    "theta": np.deg2rad(30.0),
    "x0": 50.0,
    "y0": 50.0,
}

true_sky = 5.0
true_sigma = 3.0
psf_fwhm = 3.0

pd.DataFrame({
    "truth": {
        **truth,
        "theta": np.rad2deg(truth["theta"]),
        "theta_unit": "deg",
        "sky": true_sky,
        "read_noise_sigma": true_sigma,
        "psf_fwhm": psf_fwhm,
    }
})

In [ ]:
observation = make_observation(
    galaxy=SyntheticSersic(**truth),
    shape=shape,
    psf_fwhm=psf_fwhm,
    poisson_noise=False,
    read_noise_sigma=true_sigma,
    seed=52,
)

image_path = image_dir / "single_sersic_mock_noisy.fits"
fits.writeto(
    image_path,
    observation.image + true_sky,
    overwrite=True,
)

In [ ]:
config = {
    "object": {"name": "single_sersic_mock_noisy"},
    "data": {
        "image_path": str(image_path),
        "target_position": {"x": 50, "y": 50},
        "cutout": {"mode": "fixed", "size": 101},
    },
    "outputs": {
        "root": str(output_root),
        "overwrite": True,
    },
    "preparation": {
        "background": {"method": "empirical", "box_size": 30, "filter_size": 3},
        "noise": {"method": "empirical"},
        "detection": {"n_sigma": 3.0, "n_levels": 32, "npixels": 5},
    },
    "psf": {
        "type": "gaussian",
        "gaussian": {"fwhm": psf_fwhm},
    },
    "fit": {
        "models": ["single_sersic"],
        "sampler": {
            "name": "dynesty",
            "nlive": 300,
            "sample": "rwalk",
            "dlogz": 0.01,
            "seed": 53,
            "n_processes": 1,
            "queue_size": None,
        },
        "priors": {
            "single_sersic": {
                "flux": {"min_factor": 0.5, "max_factor": 1.5},
                "re": {"min": 4.0, "max": 12.0},
                "n": {"min": 1.0, "max": 4.0},
                "q": {"min": 0.4, "max": 0.9},
                "theta": {"min": "-pi/2", "max": "pi/2"},
                "x0": {"offset": 2.0},
                "y0": {"offset": 2.0},
                "sky": {"min": -20.0, "max": 20.0},
            }
        },
    },
    "diagnostics": {
        "corner": True,
        "residual": True,
        "radial_profile": True,
        "posterior_profile_bands": True,
    },
}

In [ ]:
preparation = run_image_preparation(config)
fit = run_real_fit(config)["single_sersic"]
products = load_image_preparation_products(config)

print(f"true sky: {true_sky}")
print(f"background estimated/subtracted: {preparation.background}")
print(f"true injected sigma: {true_sigma}")
print(f"estimated noise median: {np.nanmedian(preparation.noise)}")
print(f"target label: {preparation.target_label}")
print(f"masked fraction: {preparation.contaminant_mask.mean()}")

In [ ]:
best = fit.best_parameters
expected_sky_after_preparation = true_sky - preparation.background

rows = []
for name in ["flux", "re", "n", "q", "theta", "x0", "y0"]:
    true_value = truth[name]
    best_value = best[name]
    unit = ""

    if name == "theta":
        true_value = np.rad2deg(true_value)
        best_value = np.rad2deg(best_value)
        unit = "deg"

    rows.append({
        "parameter": name,
        "truth": true_value,
        "best_ML": best_value,
        "best_minus_truth": best_value - true_value,
        "unit": unit,
    })

rows.append({
    "parameter": "sky",
    "truth": expected_sky_after_preparation,
    "best_ML": best["sky"],
    "best_minus_truth": best["sky"] - expected_sky_after_preparation,
    "unit": "",
})

pd.DataFrame(rows)

In [ ]:
plot_fit_summary(
    data=products.prepared_data,
    model=fit.best_model_image,
    noise=products.noise,
    title="Noisy mock pipeline",
)
plt.show()

pd.DataFrame.from_dict(
    {
        "chi2": fit.chi2,
        "reduced_chi2": fit.reduced_chi2,
    },
    orient="index",
    columns=["value"],
)

A good mock-pipeline run should recover the Sérsic parameters and return a reduced chi-square close to one. The fitted `sky` should be interpreted after the background subtraction performed during image preparation.